# ViT-L dense-token — calibration puis balayage (Projet_mellifere)

Ce notebook **ne fait pas glisser des crops 224×224 à stride 112**. `ortho_annotator` a déjà un détecteur dense : un seul passage avant sur une fenêtre de 784² (par défaut) donne une grille de jetons de patch à ~10 cm de résolution au sol — c'est la heatmap, en un forward pass au lieu de centaines de crops. Ce notebook ne fait que (a) changer l'encodeur pour ViT-L et (b) le faire tourner sur GPU (le code local était câblé CPU uniquement — patché dans ce commit pour accepter `--device`).

**Déroulé en 2 étapes, avec une porte de décision entre les deux :**

1. **Étape A — calibration.** `prospect learn --model dinov3-vitl16 --rasters Maison` (un seul ortho, ~14 Go, celui qui a le plus d'espèces). Ça mesure le rappel/précision dense par espèce, comparable au tableau du README (mesuré en ViT-S/16 sur les 7 orthos). **Si ViT-L ne bat pas ces chiffres, le balayage complet ci-dessous n'apporte rien — ne pas le lancer.**
2. **Étape B — balayage.** Seulement si l'étape A montre un vrai gain : `prospect scan --mode dense` sur les orthos choisis, puis export GPKG pour QGIS.

Rappels utiles (voir la conversation qui a produit ce notebook) :
- Le balayage dense est déjà **row-major séquentiel** (`prospect.py:scan_raster_dense`), cohérent avec le stockage en bandes des GeoTIFF — pas d'accès aléatoire à corriger.
- `Lotcorn`/`Leuvul` n'existent que sur l'ortho **Maison** ; `Daucar` est à 82% sur **TrailErable**. Pas besoin des 131 Go de `Dataset_Leo`/`Projet_mellifere` d'un coup — un ortho à la fois, monté depuis Drive.
- `HF_HUB_OFFLINE` est forcé à `1` par défaut dans `ortho_annotator` (poids locaux uniquement) : ce notebook le désactive explicitement pour pouvoir télécharger ViT-L depuis Hugging Face.

## 0. Runtime — vérifier le GPU
Menu *Exécution > Modifier le type d'exécution* > GPU si la cellule suivante ne montre rien.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Cloner le dépôt (privé — jeton GitHub à la volée, jamais écrit sur disque)
Un *fine-grained personal access token* avec accès lecture au dépôt `Lmague/benchmark-memoire` suffit. Laisser vide si le dépôt est public.

In [ ]:
import getpass, os

REPO = "Lmague/benchmark-memoire"
PROJECT_DIR = "/content/benchmark-memoire"

token = getpass.getpass(f"Jeton GitHub pour {REPO} (Entrée si public) : ")
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

if not os.path.isdir(PROJECT_DIR):
    !git clone --depth 1 "{url}" "{PROJECT_DIR}"
else:
    print(f"{PROJECT_DIR} existe déjà, pas de re-clone.")
del token, url  # ne pas laisser le jeton en clair dans une variable notebook

## 2. Dépendances
`torch` est déjà présent (CUDA) sur Colab. Le reste (lecture géospatiale + `transformers`) ne l'est pas.

In [ ]:
!pip install -q rasterio fiona geopandas shapely pillow scipy "transformers>=4.40"

import torch
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

## 3. Monter Drive et localiser `Projet_mellifere`
Recherche automatique sous `/content/drive` (fonctionne que le dossier soit dans *Mon Drive* ou ajouté comme raccourci depuis un partage) — **vérifier la sortie** avant de continuer, les chemins peuvent différer entre `MyDrive` et `Shareddrives`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import subprocess

hits = subprocess.run(
    ["find", "/content/drive", "-maxdepth", "5", "-iname", "Projet_mellifere", "-type", "d"],
    capture_output=True, text=True
).stdout.splitlines()
print("Candidats trouvés :")
for h in hits:
    print(" ", h)
if not hits:
    print("Aucun 'Projet_mellifere' trouvé automatiquement — fixer PROJET_ROOT à la main ci-dessous.")

In [ ]:
# À AJUSTER si la recherche ci-dessus a trouvé autre chose, ou plusieurs candidats.
PROJET_ROOT = hits[0] if hits else "/content/drive/MyDrive/Projet_mellifere"
print("PROJET_ROOT =", PROJET_ROOT)

!find "{PROJET_ROOT}/Orthomosaiques" -maxdepth 1 -type f | sort

## 4. Chemins et paramètres
`RASTER_MAISON` suppose la même arborescence/nommage que `Dataset_Leo`. **Corriger si la sortie de la cellule précédente montre un nom différent.**

In [ ]:
from pathlib import Path

EXISTING_ANNOTATIONS = f"{PROJET_ROOT}/Orthomosaiques/Annotations.gpkg"
RASTER_DIR = f"{PROJET_ROOT}/Orthomosaiques"
RASTER_MAISON = "Orthom_Maison_9Aout23_WGS84UTM18N.tif"  # cf. audit_dataset_leo.md — Lotcorn/Leuvul y sont exclusifs

# Sortie du run Colab : sur Drive, PAS sous Dataset_Leo/Projet_mellifere (l'outil refuse d'écrire là).
RUN_DIR = f"{PROJET_ROOT}/colab_runs"
OUTPUT_GPKG = f"{RUN_DIR}/session_vitl.gpkg"
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)

MODEL_ID = "facebook/dinov3-vitl16-pretrain-lvd1689m"

assert Path(EXISTING_ANNOTATIONS).is_file(), f"introuvable : {EXISTING_ANNOTATIONS}"
assert (Path(RASTER_DIR) / RASTER_MAISON).is_file(), f"introuvable : {RASTER_DIR}/{RASTER_MAISON}"
print("OK — chemins valides.")

In [ ]:
import os

# ortho_annotator force HF_HUB_OFFLINE=1 par défaut (poids locaux uniquement, machine
# d'annotation sans réseau). Sur Colab on VEUT télécharger ViT-L : on désactive avant tout
# import du package (setdefault() ne touche pas une variable déjà positionnée).
os.environ["HF_HUB_OFFLINE"] = "0"
os.environ["TRANSFORMERS_OFFLINE"] = "0"

%cd {PROJECT_DIR}/tools/ortho_annotator
import sys
sys.path.insert(0, ".")

## 5. Étape A — calibration ViT-L sur Maison
Mêmes `--span-m 5 --side-px 784` que les chiffres du README (ViT-S/16, 7 orthos) : seul le modèle change, pour isoler son effet. Coût attendu : quelques minutes sur GPU (5 espèces présentes sur Maison, `--windows-per-species 6` par défaut + fenêtres de calibration réservées).

In [ ]:
!python -m ortho_annotator prospect learn \
  --output "{OUTPUT_GPKG}" \
  --existing-annotations "{EXISTING_ANNOTATIONS}" \
  --raster-dir "{RASTER_DIR}" \
  --rasters "{RASTER_MAISON}" \
  --model "{MODEL_ID}" \
  --device cuda \
  --span-m 5.0 --side-px 784

In [ ]:
import json

cal = json.load(open(f"{RUN_DIR}/prospect/calibration.json"))

# Chiffres du README (ViT-S/16, 7 orthomosaïques, span 5 m / 784 px) — étage dense uniquement.
readme_dense = {
    "Ascsyr": (0.63, 0.35), "Daucar": (0.57, 0.50), "Eumac": (0.63, 0.50),
    "Lotcorn": (0.86, 0.61), "Solcan": (0.96, 0.78),
}

print(f"{'espèce':10s} {'ViT-S dense R/P (README, 7 orthos)':>36s}   {'ViT-L dense R/P (Maison seul)':>32s}")
for code, (r0, p0) in sorted(readme_dense.items()):
    d = cal.get("dense", {}).get(code)
    here = f"{d['recall']:.2f}/{d['precision']:.2f}" if d else "— (absent de Maison)"
    print(f"{code:10s} {r0:.2f}/{p0:.2f} {'':>28s}   {here:>32s}")

print("\nAttention : comparaison Maison-seul (ViT-L) vs 7-orthos (ViT-S) — indicatif, pas"
      " apples-to-apples. Si la tendance ne bouge pas nettement, ViT-L n'apporte probablement rien.")

## Porte de décision
**Ne pas exécuter la suite si les chiffres ci-dessus ne s'améliorent pas.** Reste sur l'outil ViT-S local (`ortho_annotator prospect scan --mode auto`, README) plutôt que de payer le transfert + le compute Colab pour rien — cohérent avec le fait que la taille de modèle n'a pas fait de différence sur la bench 12 modèles du projet.

## 6. Étape B — balayage dense (heatmap de candidats)
`SMOKE_TEST=True` d'abord : `--max-windows` borne le balayage à quelques dizaines de fenêtres pour vérifier que tout tourne (quelques secondes) avant de lancer l'ortho en entier. Passer à `False` seulement après un smoke test réussi.

In [ ]:
SMOKE_TEST = True
MAX_WINDOWS = 50 if SMOKE_TEST else 0  # 0 = pas de limite

# Un ortho à la fois : Lotcorn/Leuvul et le gros de Eumac/Solcan sont sur Maison ;
# ajouter TrailErable (Daucar) séparément une fois Maison validé.
RASTERS_TO_SCAN = [RASTER_MAISON]

for name in RASTERS_TO_SCAN:
    raster_path = f"{RASTER_DIR}/{name}"
    !python -m ortho_annotator prospect scan \
      --output "{OUTPUT_GPKG}" \
      --raster "{raster_path}" \
      --existing-annotations "{EXISTING_ANNOTATIONS}" \
      --mode dense \
      --model "{MODEL_ID}" \
      --device cuda \
      --max-windows {MAX_WINDOWS}

Le résultat est déjà persisté dans `{RUN_DIR}/prospect/candidates.sqlite` (sur Drive) — pas de session Colab de plusieurs heures à protéger d'une déconnexion pour ce qui est déjà scanné. Le balayage n'est en revanche **pas repris fenêtre par fenêtre** : une déconnexion en cours d'ortho oblige à relancer cet ortho depuis le début (d'où : un ortho par cellule/tour de boucle, pas les 7 d'un coup).

## 7. Export QGIS + aperçu rapide

In [ ]:
CANDIDATES_GPKG = f"{RUN_DIR}/candidats_vitl.gpkg"
!python -m ortho_annotator prospect export \
  --output "{OUTPUT_GPKG}" \
  --dest "{CANDIDATES_GPKG}"

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

gdf = gpd.read_file(CANDIDATES_GPKG)
print(gdf["label"].value_counts() if "label" in gdf.columns else gdf.head())

fig, ax = plt.subplots(figsize=(8, 8))
gdf.plot(ax=ax, column="label" if "label" in gdf.columns else None,
         markersize=4, legend=True, cmap="tab10")
ax.set_title("Candidats ViT-L dense — aperçu (ouvrir le .gpkg dans QGIS pour le détail)")
ax.set_aspect("equal")
plt.show()